# 06 — CLV Classification Model
## Customer Analytics Platform

Purpose: Train an XGBoost model to predict
which customers are likely to become high value.
Features: RFM scores (recency, frequency, monetary)
Target: is_high_value (1 = High, 0 = Medium/Low)
Track all experiments using MLflow.

In [0]:
# install required libraries

%pip install xgboost mlflow scikit-learn

In [0]:
# restart kernel to use updated packages
dbutils.library.restartPython()

In [0]:
import xgboost as xgb
import mlflow
import sklearn

print(f"xgboost  : {xgb.__version__}")
print(f"mlflow   : {mlflow.__version__}")
print(f"sklearn  : {sklearn.__version__}")
print("all libraries ready")

In [0]:
# load gold layer rfm features for model training

gold_path = "/Volumes/workspace/default/olist_raw_data/gold"

gold_df = spark.read.format("delta") \
    .load(f"{gold_path}/rfm_features")

print(f"total customers : {gold_df.count()}")
print("\nsample data:")
gold_df.select(
    "customer_unique_id",
    "recency",
    "frequency", 
    "monetary",
    "rfm_total_score",
    "customer_segment",
    "is_high_value"
).show(5)

In [0]:
# convert spark dataframe to pandas for sklearn and xgboost

import pandas as pd

# select only features and target
pdf = gold_df.select(
    "recency",
    "frequency",
    "monetary",
    "r_score",
    "f_score",
    "m_score",
    "rfm_total_score",
    "is_high_value"
).toPandas()

# fill any remaining nulls
pdf = pdf.fillna(0)

print(f"dataset shape : {pdf.shape}")
print(f"\nclass distribution:")
print(pdf["is_high_value"].value_counts())
print(f"\nnull values:")
print(pdf.isnull().sum())

In [0]:
# split data into train and test sets

from sklearn.model_selection import train_test_split

# define features and target
X = pdf[[
    "recency",
    "frequency",
    "monetary",
    "r_score",
    "f_score",
    "m_score",
    "rfm_total_score"
]]

y = pdf["is_high_value"]

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print(f"training set   : {X_train.shape[0]} rows")
print(f"test set       : {X_test.shape[0]} rows")
print(f"\ntrain class distribution:")
print(y_train.value_counts())
print(f"\ntest class distribution:")
print(y_test.value_counts())

In [0]:
# train xgboost model and track with mlflow

import mlflow
import mlflow.xgboost
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# set mlflow experiment with full workspace path
mlflow.set_experiment(
    "/Users/elitahazelgorimanikonda@gmail.com/CLV_Customer_Segmentation"
)

with mlflow.start_run(run_name="xgboost_clv_v1"):
    
    # define model
    model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss",
        verbosity=0
    )
    
    # train model
    model.fit(X_train, y_train)
    
    # predict
    y_pred      = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    
    # calculate metrics
    accuracy  = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)
    f1        = f1_score(y_test, y_pred)
    roc_auc   = roc_auc_score(y_test, y_pred_prob)
    
    # log parameters to mlflow
    mlflow.log_param("n_estimators",  100)
    mlflow.log_param("max_depth",     4)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("train_size",    X_train.shape[0])
    mlflow.log_param("test_size",     X_test.shape[0])
    
    # log metrics to mlflow
    mlflow.log_metric("accuracy",  accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall",    recall)
    mlflow.log_metric("f1_score",  f1)
    mlflow.log_metric("roc_auc",   roc_auc)
    
    # log model to mlflow
    mlflow.xgboost.log_model(model, "xgboost_clv_model")
    
    # print results
    print("Model Training Complete")
    print("-" * 40)
    print(f"accuracy  : {accuracy:.4f}")
    print(f"precision : {precision:.4f}")
    print(f"recall    : {recall:.4f}")
    print(f"f1 score  : {f1:.4f}")
    print(f"roc auc   : {roc_auc:.4f}")
    print("-" * 40)
    print("experiment logged to mlflow")

In [0]:
# retrain using only raw rfm values to avoid data leakage
# remove r_score, f_score, m_score, rfm_total_score

X_raw = pdf[[
    "recency",
    "frequency",
    "monetary"
]]

y = pdf["is_high_value"]

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_raw, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

mlflow.set_experiment(
    "/Users/elitahazelgorimanikonda@gmail.com/CLV_Customer_Segmentation"
)

with mlflow.start_run(run_name="xgboost_clv_v2_no_leakage"):
    
    model2 = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss",
        verbosity=0
    )
    
    model2.fit(X_train2, y_train2)
    
    y_pred2      = model2.predict(X_test2)
    y_pred_prob2 = model2.predict_proba(X_test2)[:, 1]
    
    accuracy2  = accuracy_score(y_test2, y_pred2)
    precision2 = precision_score(y_test2, y_pred2)
    recall2    = recall_score(y_test2, y_pred2)
    f12        = f1_score(y_test2, y_pred2)
    roc_auc2   = roc_auc_score(y_test2, y_pred_prob2)
    
    mlflow.log_param("features",      "recency, frequency, monetary")
    mlflow.log_param("n_estimators",  100)
    mlflow.log_param("max_depth",     4)
    mlflow.log_param("learning_rate", 0.1)
    
    mlflow.log_metric("accuracy",  accuracy2)
    mlflow.log_metric("precision", precision2)
    mlflow.log_metric("recall",    recall2)
    mlflow.log_metric("f1_score",  f12)
    mlflow.log_metric("roc_auc",   roc_auc2)
    
    mlflow.xgboost.log_model(model2, "xgboost_clv_v2")
    
    print("Model v2 Training Complete (no leakage)")
    print("-" * 40)
    print(f"features  : recency, frequency, monetary")
    print(f"accuracy  : {accuracy2:.4f}")
    print(f"precision : {precision2:.4f}")
    print(f"recall    : {recall2:.4f}")
    print(f"f1 score  : {f12:.4f}")
    print(f"roc auc   : {roc_auc2:.4f}")
    print("-" * 40)
    print("v2 experiment logged to mlflow")

In [0]:
# plot feature importance to understand which features drive predictions

import matplotlib.pyplot as plt

feature_names = ["recency", "frequency", "monetary"]
importance    = model2.feature_importances_

plt.figure(figsize=(8, 4))
plt.barh(feature_names, importance)
plt.xlabel("importance score")
plt.title("XGBoost Feature Importance — CLV Model")
plt.tight_layout()
plt.show()

print("\nfeature importance scores:")
for name, score in zip(feature_names, importance):
    print(f"  {name:<12} : {score:.4f}")